# Dataset Split and Organization Notebook

## Overview

This notebook organizes annotated image patches into train/validation/test splits for model training. It supports two workflows:

1. **Images + Labels**: Split both image patches and corresponding YOLO-format label files
2. **Images Only**: Split image patches without labels (for unlabeled datasets)

The notebook uses stratified random splitting to ensure representative distribution across splits.

### Workflow
1. Load a dataset and locate augmented images and labels
2. Create output directory structure
3. Perform train/validation/test split (80/10/10 by default)
4. Copy files to organized directories following YOLO dataset conventions

### Output Structure
```
final_Data_set/
├── images/
│   ├── train/     (80% of data)
│   ├── val/       (10% of data)
│   └── test/      (10% of data, optional)
└── labels/
    ├── train/
    ├── val/
    └── test/      (optional)
```

### Requirements
- scikit-learn (`sklearn.model_selection`)
- NumPy, OpenCV
- `CellProcessor` module (dataset utilities)

In [9]:
"""Import required libraries for dataset management and splitting."""
import os
import shutil
from sklearn.model_selection import train_test_split
from CellProcessor import (
    use_dataset,
    list_dataset
)


def copy_files_to_directory(file_paths, destination_dir):
    """
    Copy files to destination directory.
    
    Args:
        file_paths (list): Absolute paths to source files
        destination_dir (str): Destination directory (will be created if not exists)
    
    Raises:
        IOError: If file copy fails
    """
    try:
        os.makedirs(destination_dir, exist_ok=True)
    except OSError as e:
        print(f"⚠ Warning: {e}")
    
    for src_path in file_paths:
        try:
            shutil.copy2(src_path, destination_dir)
        except Exception as e:
            print(f"✗ Error copying {os.path.basename(src_path)}: {e}")
            raise

In [10]:
"""List available datasets and select one to use."""
print("Available datasets:")
list_dataset()

Available datasets:


,ID,Cell_type,Death_type,Image_path,Description
0,1,MEF,Necroptosis,Data,Test


## Load Dataset Configuration

Load dataset metadata and construct paths to augmented images and labels.

## Split Images + Labels

Split both image patches and YOLO-format label files into train/validation/test sets.

**Default split: 80% train, 10% validation, 10% test**

In [11]:
"""Load dataset configuration and construct input paths."""
# Load dataset (using preset 1; modify as needed)
dataset = use_dataset(1)
print(f"Dataset: {dataset['Cell_type']} cells, {dataset['Death_type']} death type")
print(f"Image path: {dataset['Image_path']}\n")

# Construct input paths (source: augmented data)
base_path = os.path.join(dataset['Image_path'], dataset['Death_type'])
IMAGES_DIR_PATH = os.path.join(base_path, f"{dataset['Cell_type']}_Phase_Crop_aug/")
LABELS_DIR_PATH = os.path.join(base_path, f"{dataset['Cell_type']}_Labeled_phase_aug/")

print(f"Source directories:")
print(f"  Images: {IMAGES_DIR_PATH}")
print(f"  Labels: {LABELS_DIR_PATH}")

Dataset: MEF cells, Necroptosis death type
Image path: Data

Source directories:
  Images: Data/Necroptosis/MEF_Phase_Crop_aug/
  Labels: Data/Necroptosis/MEF_Labeled_phase_aug/


In [17]:
"""
Prepare images and labels for splitting.

Match each image file with its corresponding YOLO label file (same base name).
"""

# Collect image files (filter by extension)
image_files = sorted([
    os.path.join(IMAGES_DIR_PATH, f)
    for f in os.listdir(IMAGES_DIR_PATH)
    if f.lower().endswith(('.png', '.jpg', '.jpeg'))
])

# Create corresponding label file paths
label_files = [
    os.path.join(LABELS_DIR_PATH, os.path.splitext(os.path.basename(img))[0] + ".txt")
    for img in image_files
]

# Validate matching pairs
print(f"Images found:  {len(image_files)}")
print(f"Labels found:  {len(label_files)}")

if len(image_files) != len(label_files):
    print("⚠ Warning: Image and label counts don't match")

# Perform train/validation/test split (80/10/10)
print("\n--- Performing Train/Val/Test Split (80/10/10) ---\n")

# First split: 80% train, 20% temp (val + test)
train_images, temp_images, train_labels, temp_labels = train_test_split(
    image_files, label_files,
    test_size=0.2,
    random_state=99
)

# Second split: Split temp (20%) into val (50%) and test (50%)
val_images, test_images, val_labels, test_labels = train_test_split(
    temp_images, temp_labels,
    test_size=0.5,
    random_state=99
)

# Print split statistics
print(f"Train: {len(train_images)} images, {len(train_labels)} labels")
print(f"Val:   {len(val_images)} images, {len(val_labels)} labels")
print(f"Test:  {len(test_images)} images, {len(test_labels)} labels")
print(f"Total: {len(train_images) + len(val_images) + len(test_images)} images")

Images found:  3042
Labels found:  3042

--- Performing Train/Val/Test Split (80/10/10) ---

Train: 2433 images, 2433 labels
Val:   304 images, 304 labels
Test:  305 images, 305 labels
Total: 3042 images


In [18]:
"""Copy files to organized directory structure."""

# Construct output base path
output_base = os.path.join(dataset['Image_path'], "final_Data_set")

print(f"\n--- Copying Files to Output Directory ---\n")
print(f"Output base: {output_base}\n")

# Define output directories
output_dirs = {
    "train": (output_base + "/images/train/", output_base + "/labels/train/"),
    "val":   (output_base + "/images/val/",   output_base + "/labels/val/"),
    "test":  (output_base + "/images/test/",  output_base + "/labels/test/")
}

# Copy train split
print("Copying train split...")
copy_files_to_directory(train_images, output_dirs["train"][0])
copy_files_to_directory(train_labels, output_dirs["train"][1])
print(f"  ✓ Train: {len(train_images)} images, {len(train_labels)} labels\n")

# Copy validation split
print("Copying validation split...")
copy_files_to_directory(val_images, output_dirs["val"][0])
copy_files_to_directory(val_labels, output_dirs["val"][1])
print(f"  ✓ Val: {len(val_images)} images, {len(val_labels)} labels\n")

# Copy test split
print("Copying test split...")
copy_files_to_directory(test_images, output_dirs["test"][0])
copy_files_to_directory(test_labels, output_dirs["test"][1])
print(f"  ✓ Test: {len(test_images)} images, {len(test_labels)} labels\n")

print("=" * 60)
print("SPLIT AND ORGANIZATION COMPLETE")
print("=" * 60)
print(f"Output directory: {output_base}")
print(f"\nDirectory structure:")
print(f"  {output_base}/images/train/  ({len(train_images)} files)")
print(f"  {output_base}/images/val/    ({len(val_images)} files)")
print(f"  {output_base}/images/test/   ({len(test_images)} files)")
print(f"  {output_base}/labels/train/  ({len(train_labels)} files)")
print(f"  {output_base}/labels/val/    ({len(val_labels)} files)")
print(f"  {output_base}/labels/test/   ({len(test_labels)} files)")
print("=" * 60)


--- Copying Files to Output Directory ---

Output base: Data/final_Data_set

Copying train split...
  ✓ Train: 2433 images, 2433 labels

Copying validation split...
  ✓ Val: 304 images, 304 labels

Copying test split...
  ✓ Test: 305 images, 305 labels

SPLIT AND ORGANIZATION COMPLETE
Output directory: Data/final_Data_set

Directory structure:
  Data/final_Data_set/images/train/  (2433 files)
  Data/final_Data_set/images/val/    (304 files)
  Data/final_Data_set/images/test/   (305 files)
  Data/final_Data_set/labels/train/  (2433 files)
  Data/final_Data_set/labels/val/    (304 files)
  Data/final_Data_set/labels/test/   (305 files)


## Alternative: Split Images Only if needed

Split image patches without labels (for unlabeled datasets or additional processing).

**Default split: 80% train, 10% validation, 10% test**

In [20]:

# OPTIONAL: Split images only (images-only workflow)

# Collect image files
image_files = sorted([
    os.path.join(IMAGES_DIR_PATH, f)
    for f in os.listdir(IMAGES_DIR_PATH)
    if f.lower().endswith(('.png', '.jpg', '.jpeg'))
])

print(f"Images found: {len(image_files)}")

# Perform train/validation/test split (80/10/10)
print("\n--- Performing Train/Val/Test Split (80/10/10) ---\n")

# First split: 80% train, 20% temp (val + test)
train_images, temp_images = train_test_split(
    image_files,
    test_size=0.2,
    random_state=42
)

# Second split: Split temp (20%) into val (50%) and test (50%)
val_images, test_images = train_test_split(
    temp_images,
    test_size=0.5,
    random_state=42
)

# Print split statistics
print(f"Train: {len(train_images)} images")
print(f"Val:   {len(val_images)} images")
print(f"Test:  {len(test_images)} images")
print(f"Total: {len(train_images) + len(val_images) + len(test_images)} images")


Images found: 3042

--- Performing Train/Val/Test Split (80/10/10) ---

Train: 2433 images
Val:   304 images
Test:  305 images
Total: 3042 images


In [21]:

# OPTIONAL: Copy images only to output directories

output_base = os.path.join(dataset['Image_path'], "final_Data_set")
print(f"\n--- Copying Images to Output Directory ---\n")
print(f"Output base: {output_base}\n")

copy_files_to_directory(train_images, output_base + "/images/train/")
copy_files_to_directory(val_images, output_base + "/images/val/")
copy_files_to_directory(test_images, output_base + "/images/test/")

print("=" * 60)
print("SPLIT AND ORGANIZATION COMPLETE (IMAGES ONLY)")
print("=" * 60)
print(f"Output directory: {output_base}")
print(f"  {output_base}/images/train/  ({len(train_images)} files)")
print(f"  {output_base}/images/val/    ({len(val_images)} files)")
print(f"  {output_base}/images/test/   ({len(test_images)} files)")
print("=" * 60)

print("(Images-only copy section commented out. Use main workflow above.)")


--- Copying Images to Output Directory ---

Output base: Data/final_Data_set

SPLIT AND ORGANIZATION COMPLETE (IMAGES ONLY)
Output directory: Data/final_Data_set
  Data/final_Data_set/images/train/  (2433 files)
  Data/final_Data_set/images/val/    (304 files)
  Data/final_Data_set/images/test/   (305 files)
(Images-only copy section commented out. Use main workflow above.)
